In [4]:
from pathlib import Path

import polars as pl
from climate_indices import indices
# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent.resolve()

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "observations"
    / "master_observations.parquet"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "spi"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SPI3_FILE = OUTPUT_DIR / "spi3.parquet"
SPI6_FILE = OUTPUT_DIR / "spi6.parquet"
SPI12_FILE = OUTPUT_DIR / "spi12.parquet"

print("=" * 60)
print("BUILD SPI DATASETS")
print("=" * 60)
print(f"Input : {INPUT_FILE}")
print(f"Output: {OUTPUT_DIR}")
print("=" * 60)

assert INPUT_FILE.exists()

BUILD SPI DATASETS
Input : C:\Projects\ews\zoryn\data\processed\observations\master_observations.parquet
Output: C:\Projects\ews\zoryn\data\processed\spi


In [5]:
# ============================================================
# LOAD OBSERVATIONS
# ============================================================

observations = (
    pl.scan_parquet(INPUT_FILE)
    .filter(
        pl.col("variable") == "precipitation_amount"
    )
    .sort(
        [
            "h3_id",
            "timestamp",
        ]
    )
)

summary = (
    observations
    .select(
        [
            pl.len().alias("rows"),
            pl.n_unique("h3_id").alias("h3_cells"),
            pl.n_unique("timestamp").alias("timestamps"),
            pl.min("timestamp").alias("first_timestamp"),
            pl.max("timestamp").alias("last_timestamp"),
        ]
    )
    .collect()
)

print("=" * 60)
print("OBSERVATION SUMMARY")
print("=" * 60)
print(summary)
print("=" * 60)

OBSERVATION SUMMARY
shape: (1, 5)
┌──────────┬──────────┬────────────┬─────────────────┬────────────────┐
│ rows     ┆ h3_cells ┆ timestamps ┆ first_timestamp ┆ last_timestamp │
│ ---      ┆ ---      ┆ ---        ┆ ---             ┆ ---            │
│ u32      ┆ u32      ┆ u32        ┆ str             ┆ str            │
╞══════════╪══════════╪════════════╪═════════════════╪════════════════╡
│ 11679216 ┆ 42316    ┆ 276        ┆ 2002-01         ┆ 2024-12        │
└──────────┴──────────┴────────────┴─────────────────┴────────────────┘


In [6]:
# ============================================================
# BUILD MONTHLY TIME SERIES
# ============================================================

observations = (
    observations
    .select(
        [
            "h3_id",
            "timestamp",
            "value",
        ]
    )
)

counts = (
    observations
    .group_by("h3_id")
    .agg(
        pl.len().alias("months")
    )
    .collect()
)

print("=" * 60)
print("TIME SERIES QA")
print("=" * 60)
print(counts.head())
print()
print(f"Min months : {counts['months'].min()}")
print(f"Max months : {counts['months'].max()}")
print("=" * 60)

assert counts["months"].min() == 276
assert counts["months"].max() == 276

TIME SERIES QA
shape: (5, 2)
┌─────────────────┬────────┐
│ h3_id           ┆ months │
│ ---             ┆ ---    │
│ str             ┆ u32    │
╞═════════════════╪════════╡
│ 8a7a6e50e8cffff ┆ 276    │
│ 8a7a6e5a8447fff ┆ 276    │
│ 8a7a6e55b42ffff ┆ 276    │
│ 8a7a6e51d087fff ┆ 276    │
│ 8a7a6e43578ffff ┆ 276    │
└─────────────────┴────────┘

Min months : 276
Max months : 276


In [2]:
# ============================================================
# PREPARE TIME SERIES
# ============================================================

from pathlib import Path

import polars as pl

PROJECT_ROOT = Path.cwd().parent.resolve()

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "observations"
    / "master_observations.parquet"
)

observations = (
    pl.scan_parquet(INPUT_FILE)
    .filter(
        pl.col("variable") == "precipitation_amount"
    )
    .select(
        [
            "h3_id",
            "timestamp",
            "value",
        ]
    )
    .sort(
        [
            "h3_id",
            "timestamp",
        ]
    )
    .collect()
)

histories = observations.partition_by(
    "h3_id",
    maintain_order=True,
)

print("=" * 60)
print("TIME SERIES")
print("=" * 60)
print(f"H3 histories : {len(histories):,}")
print(f"Months       : {len(histories[0]):,}")
print("=" * 60)

assert len(histories) == observations["h3_id"].n_unique()
assert len(histories[0]) == 276

TIME SERIES
H3 histories : 42,316
Months       : 276


In [3]:
# ============================================================
# COMPUTE SPI
# ============================================================

import sys
from pathlib import Path

import polars as pl

PROJECT_ROOT = Path.cwd().parent.resolve()
SRC = PROJECT_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from zoryn.indices import compute_spi

spi3 = []
spi6 = []
spi12 = []

for i, history in enumerate(histories, start=1):

    rainfall = history["value"].to_numpy()

    try:
        s3 = compute_spi(
            rainfall,
            scale=3,
        )

        s6 = compute_spi(
            rainfall,
            scale=6,
        )

        s12 = compute_spi(
            rainfall,
            scale=12,
        )

    except Exception as e:
        print("=" * 60)
        print(f"FAILED HISTORY: {i:,}")
        print("=" * 60)
        print(history)
        print("=" * 60)
        raise e

    history = history.with_columns(
        [
            pl.Series("spi3", s3),
            pl.Series("spi6", s6),
            pl.Series("spi12", s12),
        ]
    )

    spi3.append(
        history.select(
            [
                "h3_id",
                "timestamp",
                "spi3",
            ]
        )
    )

    spi6.append(
        history.select(
            [
                "h3_id",
                "timestamp",
                "spi6",
            ]
        )
    )

    spi12.append(
        history.select(
            [
                "h3_id",
                "timestamp",
                "spi12",
            ]
        )
    )

    if i % 5000 == 0:
        print(f"{i:,}/{len(histories):,}")

spi3 = pl.concat(spi3)
spi6 = pl.concat(spi6)
spi12 = pl.concat(spi12)

print("=" * 60)
print("SPI COMPLETE")
print("=" * 60)
print(f"SPI-3  : {len(spi3):,}")
print(f"SPI-6  : {len(spi6):,}")
print(f"SPI-12 : {len(spi12):,}")
print("=" * 60)

5,000/42,316
10,000/42,316
15,000/42,316
20,000/42,316
25,000/42,316
30,000/42,316
35,000/42,316
40,000/42,316
SPI COMPLETE
SPI-3  : 11,679,216
SPI-6  : 11,679,216
SPI-12 : 11,679,216


In [4]:
# ============================================================
# SAVE SPI
# ============================================================

from pathlib import Path

import polars as pl

PROJECT_ROOT = Path.cwd().parent.resolve()

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "spi"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

spi3.write_parquet(
    OUTPUT_DIR / "spi3.parquet"
)

spi6.write_parquet(
    OUTPUT_DIR / "spi6.parquet"
)

spi12.write_parquet(
    OUTPUT_DIR / "spi12.parquet"
)

print("=" * 60)
print("SPI SAVED")
print("=" * 60)
print(OUTPUT_DIR / "spi3.parquet")
print(OUTPUT_DIR / "spi6.parquet")
print(OUTPUT_DIR / "spi12.parquet")
print("=" * 60)

SPI SAVED
C:\Projects\ews\zoryn\data\processed\spi\spi3.parquet
C:\Projects\ews\zoryn\data\processed\spi\spi6.parquet
C:\Projects\ews\zoryn\data\processed\spi\spi12.parquet
